In [2]:
from selenium import webdriver  
import time  
from selenium.webdriver.common.keys import Keys  

driver = webdriver.Chrome()

In [3]:
# Navigate to LinkedIn
driver.get("https://www.linkedin.com")

# Wait a few seconds for the page to load
time.sleep(5)

# Print the page title to confirm
print("Page title:", driver.title)

Page title: LinkedIn: Log In or Sign Up


In [6]:
import pandas as pd

# Read the Excel file
file_path = "/home/jain/Desktop/ws/public/company_and_job_market_research/ratings_and_reviews_at_ambitionbox.xlsx"
df = pd.read_excel(file_path)

# Display basic info
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

Shape: (76, 33)

Columns: ['Date', 'Company', 'Rating', '#Reviews', "Industry level insights (%: -100 (below industry's average) to 100)", 'Sample Avg Insights', 'URL', 'Company Culture', 'Salary', 'Work-Life Balance', 'Work Satisfaction', 'Skill Development', 'Job Security', 'Promotions', 'Founded in', 'India Employee Count', 'Global Employee Count', 'Company Website', 'Company Industry', 'Other Industries', 'HQ', 'LinkedIn URL', 'Annual Revenue as of Jun 2026', 'Position', 'Avg Salary (L/Yr)', 'Salary_URL', 'My YOE', 'Median current employee tenure (in years)', 'Employee Count Growth 6M %', 'Employee Count Growth 1Y %', 'Employee Count Growth 2Y %', 'Hiring Trend (6M) – Date Sensitive Metric', 'New Hires (Past 6M)']

First 5 rows:


,Date,Company,Rating,#Reviews,Industry level insights (%: -100 (below industry's average) to 100),Sample Avg Insights,URL,Company Culture,Salary,Work-Life Balance,...,Position,Avg Salary (L/Yr),Salary_URL,My YOE,Median current employee tenure (in years),Employee Count Growth 6M %,Employee Count Growth 1Y %,Employee Count Growth 2Y %,Hiring Trend (6M) – Date Sensitive Metric,New Hires (Past 6M)
0,Jan / 16 / 2026,Axtria,2.8,748,-22,-24,https://www.ambitionbox.com/reviews/axtria-rev...,2.6,2.9,2.6,...,Lead Data Scientist,24.5,https://www.ambitionbox.com/salaries/axtria-sa...,13,3.9,0,1,0,-65,275
1,Jan / 16 / 2026,Infosys,3.5,47300,-3,-5,https://www.ambitionbox.com/reviews/infosys-re...,3.6,2.7,3.6,...,Lead Data Scientist,24.3,https://www.ambitionbox.com/salaries/infosys-s...,13,4.2,0,5,14,-30,18644
2,Jan / 16 / 2026,Accenture,3.7,71100,3,0,https://www.ambitionbox.com/reviews/accenture-...,3.7,3.2,3.6,...,Lead Data Scientist,26.9,https://www.ambitionbox.com/salaries/accenture...,13,4.0,3,6,12,-23,55714
3,Jan / 16 / 2026,Cognizant,3.6,59900,0,-2,https://www.ambitionbox.com/reviews/cognizant-...,3.5,3.2,3.6,...,Lead Data Scientist,28.3,https://www.ambitionbox.com/salaries/cognizant...,13,4.3,1,3,9,-42,24573
4,Jan / 16 / 2026,Nagarro,3.9,4700,8,6,https://www.ambitionbox.com/reviews/nagarro-re...,3.9,3.7,3.6,...,Lead Data Scientist,39.6,https://www.ambitionbox.com/salaries/nagarro-s...,13,3.8,3,6,10,-44,1502


In [ ]:
linkedin_related_columns = [
    "LinkedIn URL",
    "Median current employee tenure (in years)",
    "Employee Count Growth 6M %",
    "Employee Count Growth 1Y %",
    "Employee Count Growth 2Y %",
    "Hiring Trend (6M) – Date Sensitive Metric",
    "New Hires (Past 6M)"
]

In [7]:
import json
import re
from datetime import datetime, timedelta

def extract_linkedin_insights(driver, linkedin_url):
    """
    Navigate to a LinkedIn company Insights page and extract:
      - Employee Count Growth 6M %
      - Employee Count Growth 1Y %
      - Employee Count Growth 2Y %
      - Hiring Trend (6M)
      - New Hires (Past 6M)
      - Median current employee tenure (in years)
      - Total Employees
    """
    # Ensure we're on the Insights tab
    url = linkedin_url.rstrip("/")
    if "/insights" not in url:
        url = url + "/insights/"
    else:
        url = url if url.endswith("/") else url + "/"
    
    print(f"Navigating to: {url}")
    driver.get(url)
    time.sleep(6)
    
    # Find all <code> elements (hidden JSON data blocks)
    code_elements = driver.find_elements("tag name", "code")
    
    insights_json = None
    for code_el in code_elements:
        try:
            text = code_el.get_attribute("textContent") or code_el.text
            if "premiumDashCompanyInsightsCardByCompany" in text:
                insights_json = json.loads(text)
                print("✓ Found insights JSON block")
                break
        except:
            continue
    
    if not insights_json:
        print("✗ Could not find insights data. The page may require login or the structure changed.")
        # Print page source snippet for debugging
        print("\nPage text sample (first 2000 chars):")
        print(driver.find_element("tag name", "body").text[:2000])
        return None
    
    # Parse the JSON
    try:
        elements = insights_json["data"]["data"]["premiumDashCompanyInsightsCardByCompany"]["elements"]
        if not elements:
            print("✗ No insight elements found")
            return None
        
        company_insights = elements[0]["companyInsights"]
        hc = company_insights.get("headcountInsights", {})
        
        result = {}
        
        # 1. Growth percentages from growthPeriods
        growth_periods = hc.get("growthPeriods", [])
        for gp in growth_periods:
            months = gp["monthDifference"]
            pct = gp["changePercentage"]
            if months == 6:
                result["Employee Count Growth 6M %"] = pct
            elif months == 12:
                result["Employee Count Growth 1Y %"] = pct
            elif months == 24:
                result["Employee Count Growth 2Y %"] = pct
        
        # 2. Total employees
        result["Total Employees"] = hc.get("totalEmployees", None)
        
        # 3. Compute New Hires (Past 6M) from monthly headcount data
        headcount_growth = hc.get("headcounts", {}).get("headcountGrowth", [])
        if len(headcount_growth) >= 7:
            # Sort by date
            headcount_growth.sort(key=lambda x: (x["startedOn"]["year"], x["startedOn"]["month"]))
            latest = headcount_growth[-1]["employeeCount"]
            six_months_ago = headcount_growth[-7]["employeeCount"]
            result["New Hires (Past 6M)"] = max(0, latest - six_months_ago)
            
            # 4. Hiring Trend (6M) - month-over-month changes in the last 6 months
            last_6 = headcount_growth[-7:]  # last 7 entries = 6 month-over-month changes
            mom_changes = []
            for i in range(1, len(last_6)):
                change = last_6[i]["employeeCount"] - last_6[i-1]["employeeCount"]
                mom_changes.append(change)
            result["Hiring Trend (6M) – Monthly Changes"] = mom_changes
            result["Hiring Trend (6M) – Avg Monthly Change"] = round(sum(mom_changes) / len(mom_changes), 1)
        
        # 5. Median tenure - might be in a different location; try to find it
        # (not always available in this API response)
        result["Median current employee tenure (in years)"] = None  # placeholder
        
        # Also extract the full monthly headcount for reference
        result["_monthly_headcount"] = [
            {"date": f"{h['startedOn']['year']}-{h['startedOn']['month']:02d}", 
             "count": h["employeeCount"]}
            for h in headcount_growth
        ]
        
        return result
        
    except Exception as e:
        print(f"✗ Error parsing JSON: {e}")
        import traceback
        traceback.print_exc()
        return None


# --- Run on the first URL ---
first_url = df["LinkedIn URL"].dropna().iloc[0]
print(f"Company: {df.iloc[0]['Company Name'] if 'Company Name' in df.columns else 'N/A'}")
print(f"URL: {first_url}\n")

result = extract_linkedin_insights(driver, first_url)

if result:
    print("\n" + "="*50)
    print("EXTRACTED METRICS:")
    print("="*50)
    for key, value in result.items():
        if not key.startswith("_"):
            print(f"  {key}: {value}")
    
    print("\n--- Monthly headcount data ---")
    for entry in result.get("_monthly_headcount", []):
        print(f"  {entry['date']}: {entry['count']:,}")

Company: N/A
URL: https://www.linkedin.com/company/axtria/insights/

Navigating to: https://www.linkedin.com/company/axtria/insights/
✓ Found insights JSON block

EXTRACTED METRICS:
  Employee Count Growth 6M %: 0
  Employee Count Growth 1Y %: -1
  Employee Count Growth 2Y %: 0
  Total Employees: 3642
  New Hires (Past 6M): 8
  Hiring Trend (6M) – Monthly Changes: [-11, -24, -1, 12, 4, 28]
  Hiring Trend (6M) – Avg Monthly Change: 1.3
  Median current employee tenure (in years): None

--- Monthly headcount data ---
  2024-07: 3,639
  2024-08: 3,668
  2024-09: 3,718
  2024-10: 3,709
  2024-11: 3,695
  2024-12: 3,683
  2025-01: 3,763
  2025-02: 3,739
  2025-03: 3,736
  2025-04: 3,747
  2025-05: 3,782
  2025-06: 3,731
  2025-07: 3,694
  2025-08: 3,651
  2025-09: 3,662
  2025-10: 3,620
  2025-11: 3,590
  2025-12: 3,579
  2026-01: 3,634
  2026-02: 3,623
  2026-03: 3,599
  2026-04: 3,598
  2026-05: 3,610
  2026-06: 3,614
  2026-07: 3,642


In [ ]:
# Loop through first 5 LinkedIn URLs and extract insights
all_results = []
urls = df["LinkedIn URL"].dropna().head(5)

for i, (idx, url) in enumerate(urls.items()):
    print(f"\n{'#'*60}")
    print(f"# [{i+1}/5] Processing row {idx}")
    print(f"{'#'*60}")
    
    company_name = df.loc[idx, "Company Name"] if "Company Name" in df.columns else f"Row {idx}"
    print(f"Company: {company_name}")
    print(f"URL: {url}")
    
    result = extract_linkedin_insights(driver, url)
    
    if result:
        result["_idx"] = idx
        result["_company"] = company_name
        result["_url"] = url
        all_results.append(result)
    
    # Brief pause between requests
    if i < len(urls) - 1:
        print("\nWaiting 3 seconds before next URL...")
        time.sleep(3)

# --- Summary of all results ---
print("\n\n" + "="*70)
print("SUMMARY: LinkedIn Insights for First 5 Companies")
print("="*70)

for r in all_results:
    print(f"\n--- {r['_company']} ---")
    for key in ["Employee Count Growth 6M %", "Employee Count Growth 1Y %", 
                "Employee Count Growth 2Y %", "Total Employees", 
                "New Hires (Past 6M)", "Hiring Trend (6M) – Avg Monthly Change"]:
        print(f"  {key}: {r.get(key, 'N/A')}")